# Mamba-CL Baseline — Split-CIFAR100 & ImageNet-R

**Purpose:** Reproduce Mamba-CL results for HippoCortex Paper 1 Table 1.  
**Record outputs here:** AA, Forgetting for each run → paste into `docs/architecture.md` Baseline Results table.  
**Kaggle settings:** GPU T4 x2 or P100 · Internet ON (for downloads) · Accelerator: GPU

---
### What this notebook does
1. Installs Mamba-CL dependencies  
2. Downloads pretrained De-focus Mamba weights  
3. Prepares CIFAR-100 in the required folder structure  
4. Runs Mamba-CL on **10-task CIFAR-100** (paper setting)  
5. Runs Mamba-CL on **20-task CIFAR-100** (our HippoCortex comparison setting)  
6. Saves results as JSON for import into HippoCortex repo

## Step 1 — Install dependencies

In [ ]:
# Kaggle already has torch 2.x + CUDA. Install the remaining packages.
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Core dependencies from Mamba-CL readme
pip('timm==1.0.9')
pip('einops==0.8.0')
pip('opencv-python==4.10.0.84')
pip('scikit-image')

# Mamba SSM — the fast CUDA kernels Mamba-CL uses internally
# On Kaggle T4/P100 with CUDA 12.x this should work directly
pip('mamba-ssm==1.2.2', 'causal-conv1d==1.4.0')

# transformers needed by mamba_block.py
pip('transformers>=4.40.0')

print('All packages installed.')

## Step 2 — Clone Mamba-CL

In [ ]:
import os

MAMBA_CL_DIR = '/kaggle/working/mamba-cl'

if not os.path.exists(MAMBA_CL_DIR):
    os.system(f'git clone https://github.com/zugexiaodui/mamba-cl {MAMBA_CL_DIR}')

os.chdir(MAMBA_CL_DIR)
print('Working directory:', os.getcwd())
print(os.listdir('.'))

## Step 3 — Download pretrained De-focus Mamba weights

This is the ImageNet-21K pretrained backbone that Mamba-CL fine-tunes.  
File size ~350MB. Required — training will crash without it.

In [ ]:
WEIGHTS_URL = 'https://github.com/OpenGVLab/De-focus-Attention-Networks/releases/download/v1.0/defocus_mamba_large_cls_21k.pth'
WEIGHTS_PATH = '/kaggle/working/defocus_mamba_large_cls_21k.pth'

if not os.path.exists(WEIGHTS_PATH):
    print('Downloading pretrained weights (~350 MB)...')
    os.system(f'wget -q --show-progress -O {WEIGHTS_PATH} "{WEIGHTS_URL}"')

print(f'Weights file: {os.path.getsize(WEIGHTS_PATH) / 1e6:.1f} MB')

## Step 4 — Prepare CIFAR-100

Mamba-CL requires CIFAR-100 in a specific folder structure:
```
data.CIFAR100/
  train/
    0/   ← class index folders (0–99)
    1/
    ...
  val/
    0/
    ...
```
We download via torchvision and convert to this structure.

In [ ]:
import torchvision
import numpy as np
from pathlib import Path
from PIL import Image

CIFAR_RAW = '/kaggle/working/cifar100_raw'
CIFAR_DIR = '/kaggle/working/data.CIFAR100'

# Download raw CIFAR-100
if not Path(CIFAR_RAW).exists():
    print('Downloading CIFAR-100...')
    torchvision.datasets.CIFAR100(root=CIFAR_RAW, train=True,  download=True)
    torchvision.datasets.CIFAR100(root=CIFAR_RAW, train=False, download=True)

def convert_cifar100_to_folder(raw_root, out_root):
    """Convert CIFAR-100 binary format → numbered class folders."""
    out_root = Path(out_root)
    for split, train in [('train', True), ('val', False)]:
        dataset = torchvision.datasets.CIFAR100(root=raw_root, train=train, download=False)
        for idx, (img_arr, label) in enumerate(zip(dataset.data, dataset.targets)):
            cls_dir = out_root / split / str(label)
            cls_dir.mkdir(parents=True, exist_ok=True)
            img = Image.fromarray(img_arr)
            img.save(cls_dir / f'{idx:05d}.png')
        print(f'  {split}: done ({len(dataset)} images)')

if not Path(CIFAR_DIR).exists():
    print('Converting CIFAR-100 to folder structure...')
    convert_cifar100_to_folder(CIFAR_RAW, CIFAR_DIR)
    print('Done.')
else:
    print('CIFAR-100 folder already exists, skipping.')

# Verify
n_train = sum(1 for _ in Path(CIFAR_DIR).glob('train/*/*.png'))
n_val   = sum(1 for _ in Path(CIFAR_DIR).glob('val/*/*.png'))
print(f'Train images: {n_train}  |  Val images: {n_val}')
assert n_train == 50000 and n_val == 10000, 'Image count mismatch!'

## Step 5 — Verify the exact task split Mamba-CL uses

**Important for HippoCortex:** The task split is seed-dependent (not a static file).  
Save it once here so Induwara can replicate it exactly in `hippocortex/data/split_cifar100.py`.

In [ ]:
import numpy as np
import json

def get_mamba_cl_task_split(n_classes, n_tasks, seed):
    """Replicates ClassIncrementalManager.__init__ from Mamba-CL exactly."""
    rng = np.random.Generator(np.random.PCG64(seed))
    classes = list(range(n_classes))
    classes_arr = np.array(classes, dtype=np.int64)
    rng.shuffle(classes_arr)                          # shuffle_classes=True (default)
    task_class_list = classes_arr.reshape(n_tasks, -1).tolist()
    return task_class_list

SEED = 2024  # Mamba-CL default seed

split_cifar100_10t = get_mamba_cl_task_split(100, 10, SEED)
split_cifar100_20t = get_mamba_cl_task_split(100, 20, SEED)
split_imagenet_r_20t = get_mamba_cl_task_split(200, 20, SEED)

splits = {
    'cifar100_10tasks_seed2024': split_cifar100_10t,
    'cifar100_20tasks_seed2024': split_cifar100_20t,
    'imagenet_r_20tasks_seed2024': split_imagenet_r_20t,
}

split_path = '/kaggle/working/mamba_cl_task_splits.json'
with open(split_path, 'w') as f:
    json.dump(splits, f, indent=2)

print('Task splits saved to:', split_path)
print('CIFAR-100 10-task split (task 0):', split_cifar100_10t[0])
print('CIFAR-100 20-task split (task 0):', split_cifar100_20t[0])

# !! DOWNLOAD THIS FILE and commit to:
# data/cifar100/task_split_seed2024.json
# data/imagenet_r/task_split_seed2024.json

## Step 6 — Run Mamba-CL on CIFAR-100 (10 tasks — paper setting)

This is the number from the **published paper**. Record it in `docs/architecture.md`.

In [ ]:
import subprocess, sys

os.chdir(MAMBA_CL_DIR)

cmd = [
    sys.executable, 'train_eval.py',
    '-d', 'cifar100',
    '-t', '10',                            # 10-task split (paper default)
    '--pretrained_path', WEIGHTS_PATH,
    '--data_root', CIFAR_DIR,
    '--null_eta', '0.95',
    '--use_null_space',
    '--seed', '2024',
    '--batch_size', '200',                 # Paper default — safe on T4 (~6.5 GB peak VRAM)
    '--eval_batch_size', '100',
    '--workers', '2',
    '--eval_workers', '2',
    '--use_amp', 'true',                   # KEEP ON — saves ~1.5 GB VRAM, paper used bfloat16
    '--epochs', '10',
]

print('Running command:')
print(' '.join(cmd))
print('\n--- Training output ---')

result = subprocess.run(cmd, capture_output=False, text=True)
print('Return code:', result.returncode)

## Step 7 — Run Mamba-CL on CIFAR-100 (20 tasks — HippoCortex comparison)

Our HippoCortex targets 20-task CIFAR-100. Run this to get the comparison number.

In [ ]:
os.chdir(MAMBA_CL_DIR)

cmd_20t = [
    sys.executable, 'train_eval.py',
    '-d', 'cifar100',
    '-t', '20',                            # 20-task split (HippoCortex comparison)
    '--pretrained_path', WEIGHTS_PATH,
    '--data_root', CIFAR_DIR,
    '--null_eta', '0.95',
    '--use_null_space',
    '--seed', '2024',
    '--batch_size', '200',                 # Paper default — safe on T4
    '--eval_batch_size', '100',
    '--workers', '2',
    '--eval_workers', '2',
    '--use_amp', 'true',
    '--epochs', '10',
]

print('Running 20-task CIFAR-100...')
result = subprocess.run(cmd_20t, capture_output=False, text=True)
print('Return code:', result.returncode)

## Step 8 — Parse and save results

Run this after each training run above. It parses stdout for accuracy numbers.

In [ ]:
import re, json

# ---- Manually fill in from the output above ----
# Look for lines like:
#   ":: ** Results of task [10]: [ class_inc_last_acc= 75.23% | class_inc_last_forg=  4.12% ] **"

results = {
    'mamba_cl_cifar100_10task': {
        'AA':  None,   # <-- fill from final task output
        'forgetting': None,
        'seed': 2024,
        'null_eta': 0.95,
        'epochs': 10,
        'batch_size': 64,
        'commit': 'HEAD',  # fill with: os.popen("git rev-parse HEAD").read().strip()
    },
    'mamba_cl_cifar100_20task': {
        'AA':  None,
        'forgetting': None,
        'seed': 2024,
        'null_eta': 0.95,
        'epochs': 10,
        'batch_size': 64,
        'commit': 'HEAD',
    },
}

# Get commit hash
os.chdir(MAMBA_CL_DIR)
commit = os.popen('git rev-parse HEAD').read().strip()
for k in results:
    results[k]['commit'] = commit

out_path = '/kaggle/working/baseline_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

print('Saved to:', out_path)
print(json.dumps(results, indent=2))
print()
print('>>> COPY THESE NUMBERS TO: docs/architecture.md  (Baseline Results table)')

## Step 9 — (Optional) VRAM troubleshooting

If you see OOM (out of memory) errors, run the cell below to check available VRAM  
and then adjust `--batch_size` and `--eval_batch_size` in Steps 6/7.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('Total VRAM:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')
print('Allocated:', torch.cuda.memory_allocated(0) / 1e9, 'GB')
print('Reserved:', torch.cuda.memory_reserved(0) / 1e9, 'GB')
print()
print('Tip: defocus_mamba_large uses 192x192 images.')
print('At batch_size=64 with AMP, expect ~8-10 GB VRAM usage.')
print('If OOM: reduce to --batch_size 32 --eval_batch_size 32')

---
## What to do with the outputs

1. Download `baseline_results.json` and `mamba_cl_task_splits.json` from Kaggle output  
2. Save splits:
   - `data/cifar100/task_split_seed2024.json`  
   - `data/imagenet_r/task_split_seed2024.json`  
3. Fill in `docs/architecture.md` Baseline Results table with AA and Forgetting values  
4. Tell Induwara to load `task_split_seed2024.json` in `hippocortex/data/split_cifar100.py`  
   instead of regenerating the split — this guarantees identical task boundaries

---
## Architecture insight for HippoCortex

**Mamba-CL is NOT a vanilla Mamba** — it uses `DefocusAttentionNetwork`, a vision Mamba  
pretrained on ImageNet-21K (350MB checkpoint). It fine-tunes only 4 parameter groups:  
`x_proj`, `out_proj.weight`, `A_log`, and `head`.  

For HippoCortex Stage 1 we build our own Mamba backbone from scratch (not pretrained)  
and train end-to-end — this is a harder setting. If our results are lower than Mamba-CL,  
mention the pretrained backbone advantage in the paper's Experimental Setup section.